In [1]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import shutil

### Creating a data.csv file from the dataset. With the following columns:
    path : Path to the video file
    label : Label/Class of the video.

In [7]:
geste = []

daten_pfad = os.listdir("data/")
labels = daten_pfad
print(daten_pfad)

['hand_turn', 'geste_1', 'Gesture02', 'geste_2', 'class_2', 'thumb_up', 'geste_0', 'ok_sign', 'Gesture01', 'class_1']


In [8]:
rooms = []

for item in daten_pfad:
 # Get all the file names
 all_rooms = os.listdir('data' + '/' +item)

 # Add them to the list
 for room in all_rooms:
    rooms.append((item, str('data' + '/' +item) + '/' + room))
    
# Build a dataframe        
df = pd.DataFrame(data=rooms, columns=['tag', 'video_name'])
print(df.head())
print(df.tail())

         tag                                    video_name
0  hand_turn  data/hand_turn/WIN_20260427_17_19_54_Pro.mp4
1  hand_turn  data/hand_turn/WIN_20260427_17_12_14_Pro.mp4
2  hand_turn  data/hand_turn/WIN_20260427_17_19_40_Pro.mp4
3  hand_turn  data/hand_turn/WIN_20260427_17_20_04_Pro.mp4
4  hand_turn  data/hand_turn/WIN_20260427_17_12_09_Pro.mp4
         tag                    video_name
376  class_1  data/class_1/class_1_026.mp4
377  class_1  data/class_1/class_1_018.mp4
378  class_1  data/class_1/class_1_019.mp4
379  class_1  data/class_1/class_1_021.mp4
380  class_1  data/class_1/class_1_010.mp4


In [9]:
df = df.rename(columns={'video_name': 'path', 'tag':'label'})
df = df.iloc[:, [1,0]]
df.to_csv("data.csv")

**Splitting the dataset into train and test**

In [10]:
train, temp = train_test_split(df, test_size=0.4, stratify=df.label,random_state=42, shuffle=True)

val, test = train_test_split(temp, test_size=0.5, stratify=temp.label,random_state=42, shuffle=True)

train.to_csv("train.csv", index=False)
val.to_csv("val.csv", index=False)

test.to_csv("test.csv", index=False)

In [11]:
done

NameError: name 'done' is not defined

In [ ]:
# TODO : create a loop for train, test, val 
csv_path = "val.csv"
source_root = "c:/Users/nikam/ML2/Handgesten"
target_root = os.path.join(source_root, "new_data", "val")

df = pd.read_csv(csv_path)

for _, row in df.iterrows():
    label = row["label"]
    src = os.path.join(source_root, row["path"])
    dst_dir = os.path.join(target_root, label)
    os.makedirs(dst_dir, exist_ok=True)

    dst = os.path.join(dst_dir, os.path.basename(src))
    shutil.copy2(src, dst)

print("Done copying train files.")

Done copying train files.


In [ ]:
train_df = pd.read_csv("train.csv")
train_df['path'] = train_df['path'].str.replace("data/", "", regex=True)
train_df.to_csv("new_data/train/train.csv", index=False)

In [ ]:
val_df = pd.read_csv("val.csv")
val_df['path'] = val_df['path'].str.replace("data/", "", regex=True)
val_df.to_csv("new_data/val/val.csv", index=False)

In [ ]:
test_df = pd.read_csv("test.csv")
test_df['path'] = test_df['path'].str.replace("data/", "", regex=True)
test_df.to_csv("new_data/test/test.csv", index=False)

## Creating DataLoaders.
https://pytorchvideo.org/docs/tutorial_classification

In [ ]:
class KineticsDataModule(pytorch_lightning.LightningDataModule):

  # Dataset configuration
  _DATA_PATH = 'new_data/'
  _CLIP_DURATION = 3  # Duration of sampled clip for each video
  _BATCH_SIZE =2
  _NUM_WORKERS = 0# Number of parallel processes fetching data

  def train_dataloader(self):
    """
    Create the Kinetics train partition from the list of video labels
    in {self._DATA_PATH}/train.csv. Add transform that subsamples and
    normalizes the video before applying the scale, crop and flip augmentations.
    """
    train_transform = Compose(
        [
        ApplyTransformToKey(
          key="video",
          transform=Compose(
              [
                UniformTemporalSubsample(8),
                Lambda(lambda x: x / 255.0),
                Normalize((0.45, 0.45, 0.45), (0.225, 0.225, 0.225)),
                RandomShortSideScale(min_size=256, max_size=320),
                RandomCrop(244),
                RandomHorizontalFlip(p=0.5),
              ]
            ),
          ),
        ]
    )
    train_dataset = pytorchvideo.data.Kinetics(
        data_path=os.path.join(self._DATA_PATH, "train","train.csv"),
        clip_sampler=pytorchvideo.data.make_clip_sampler("random", self._CLIP_DURATION),
        transform=train_transform
    )
    return torch.utils.data.DataLoader(
        train_dataset,
        batch_size=self._BATCH_SIZE,
        num_workers=self._NUM_WORKERS,
    )

  def val_dataloader(self):
    """
    Create the Kinetics validation partition from the list of video labels
    in {self._DATA_PATH}/val
    """
    val_transform = Compose([
    ApplyTransformToKey(
        key="video",
        transform=Compose([
            UniformTemporalSubsample(8),
            Lambda(lambda x: x / 255.0),
            Normalize((0.45, 0.45, 0.45), (0.225, 0.225, 0.225)),
            ShortSideScale(256),
            CenterCrop(244),
        ]),
    ),
])
    val_dataset = pytorchvideo.data.Kinetics(
        data_path=os.path.join(self._DATA_PATH,'val',"val.csv"),
        clip_sampler=pytorchvideo.data.make_clip_sampler("uniform", self._CLIP_DURATION),
        transform=val_transform,
        decode_audio=False,
    )
    return torch.utils.data.DataLoader(
        val_dataset,
        batch_size=self._BATCH_SIZE,
        num_workers=self._NUM_WORKERS,
    )
  

In [ ]:
import fvcore
print(torch.__version__)
print(torchvision.__version__)
print(pytorchvideo.__version__)
print(fvcore.__version__)

2.11.0+cpu
0.26.0+cpu
0.1.5
0.1.5.post20221221


In [ ]:
from torchvision.models.video import r3d_18, R3D_18_Weights

def make_kinetics_resnet():
    model = r3d_18(weights=R3D_18_Weights.DEFAULT)  # pretrained on Kinetics-400
    model.fc = torch.nn.Linear(model.fc.in_features, 3)  # swap head for 3 classes
    return model

OSError: Can't get source for <function legacy_get_enum at 0x00000186F185C680>. TorchScript requires source access in order to carry out compilation, make sure original .py files are available.
'binary_cross_entropy_with_logits' is being compiled since it was called from 'sigmoid_focal_loss'
  File "c:\Users\nikam\ML2\Handgesten\.venv\Lib\site-packages\fvcore\nn\focal_loss.py", line 36
    targets = targets.float()
    p = torch.sigmoid(inputs)
    ce_loss = F.binary_cross_entropy_with_logits(inputs, targets, reduction="none")
    ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~ <--- HERE
    p_t = p * targets + (1 - p) * (1 - targets)
    loss = ce_loss * ((1 - p_t) ** gamma)


In [ ]:


class VideoClassificationLightningModule(pytorch_lightning.LightningModule):
  def __init__(self):
      super().__init__()
      self.model = make_kinetics_resnet()

  def forward(self, x):
      return self.model(x)

  def training_step(self, batch, batch_idx):
      # The model expects a video tensor of shape (B, C, T, H, W), which is the
      # format provided by the dataset
      y_hat = self.model(batch["video"])

      # Compute cross entropy loss, loss.backwards will be called behind the scenes
      # by PyTorchLightning after being returned from this method.
      loss = F.cross_entropy(y_hat, batch["label"])

      # Log the train loss to Tensorboard
      self.log("train_loss", loss.item())

      return loss

  def validation_step(self, batch, batch_idx):
      y_hat = self.model(batch["video"])
      loss = F.cross_entropy(y_hat, batch["label"])
      self.log("val_loss", loss)
      return loss

  def configure_optimizers(self):
      """
      Setup the Adam optimizer. Note, that this function also can return a lr scheduler, which is
      usually useful for training video models.
      """
      return torch.optim.Adam(self.parameters(), lr=1e-4)

In [ ]:
import torchvision.models as models

def make_gesture_model():
    # MobileNetV3 — lightweight, fast on CPU, pretrained weights work fine
    model = models.mobilenet_v3_small(weights="DEFAULT")
    model.classifier[3] = torch.nn.Linear(model.classifier[3].in_features, 3)
    return model

class VideoClassificationLightningModule2(pytorch_lightning.LightningModule):
    def __init__(self):
        super().__init__()
        self.model = make_gesture_model()

    def forward(self, x):
        # x shape from pytorchvideo: (B, C, T, H, W)
        B, C, T, H, W = x.shape
        # Reshape: treat each frame independently → (B*T, C, H, W)
        x = x.permute(0, 2, 1, 3, 4).reshape(B * T, C, H, W)
        out = self.model(x)                    # (B*T, 3)
        out = out.reshape(B, T, 3).mean(dim=1) # average predictions across frames
        return out

    def training_step(self, batch, batch_idx):
        y_hat = self(batch["video"])
        loss = F.cross_entropy(y_hat, batch["label"])
        acc = (y_hat.argmax(dim=1) == batch["label"]).float().mean()
        self.log("train_loss", loss, prog_bar=True)
        self.log("train_acc", acc, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        y_hat = self(batch["video"])
        loss = F.cross_entropy(y_hat, batch["label"])
        acc = (y_hat.argmax(dim=1) == batch["label"]).float().mean()
        self.log("val_loss", loss, prog_bar=True)
        self.log("val_acc", acc, prog_bar=True)

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=1e-4)

In [ ]:
def train():
    classification_module = VideoClassificationLightningModule2()
    data_module = KineticsDataModule()

    trainer = pytorch_lightning.Trainer(
        max_epochs=20,
        accelerator="gpu" if torch.cuda.is_available() else "cpu",
        devices=1,
        log_every_n_steps=5,
    )
    trainer.fit(classification_module, data_module)

    # Save the trained model
    torch.save(classification_module.model.state_dict(), "gesture_model.pth")
    print("Model saved.")

train()